In [1]:
import pandas as pd
import numpy as np
import time

df_Enem = pd.read_csv('MICRODADOS_ENEM_2023.csv',header='infer',sep=';', low_memory=False, encoding='latin-1')

df_municipios = pd.read_csv('municipios.csv',header='infer',sep=',', low_memory=False)

print(f"📊 Dados prontos: {len(df_Enem) } Registros ENEM e {len(df_municipios)} Municipios")

# --- ESTRATÉGIA A: REDUCE-SIDE JOIN (Simulando Shuffle & Sort) ---
# O merge do pandas é eficiente, mas aqui ele simula o processo de alinhar
# duas tabelas grandes onde o sistema precisa "procurar" e "ordenar" as chaves.
start_shuffle = time.time()
resultado_shuffle = pd.merge(df_Enem, df_municipios, on='CO_MUNICIPIO_PROVA')
tempo_shuffle = time.time() - start_shuffle

print(f"⏱️ Tempo 'Reduce-side Join': {tempo_shuffle:.4f} segundos")


# --- ESTRATÉGIA B: MAP-SIDE JOIN (Simulando Broadcast & Distributed Cache) ---
# Nesta estratégia, a tabela pequena é enviada para o cache (dicionário)
# e a junção acontece instantaneamente durante o mapeamento da tabela grande.
start_map = time.time()

# 1. Simulação do Distributed Cache: enviamos a tabela menor para a RAM de todos os 'Mappers'
cache_municipios = df_municipios.set_index('CO_MUNICIPIO_PROVA')['capital'].to_dict()

# 2. Simulação do Map: o Join é feito via lookup direto (O(1)) sem mover a tabela fato pela rede
df_Enem['capital'] = df_Enem['CO_MUNICIPIO_PROVA'].map(cache_municipios)

tempo_map_csv = time.time() - start_map
print(f"Tempo 'Map-side Join' (Vetorizado): {tempo_map_csv:.4f} segundos")

print(f"\nResultado: O Map-side Join foi {tempo_shuffle/tempo_map_csv:.1f}x mais rapido!")

📊 Dados prontos: 3933955 Registros ENEM e 5571 Municipios
⏱️ Tempo 'Reduce-side Join': 1.3059 segundos
Tempo 'Map-side Join' (Vetorizado): 0.0579 segundos

Resultado: O Map-side Join foi 22.6x mais rapido!


In [1]:
import pandas as pd

df_enem = pd.read_csv('MICRODADOS_ENEM_2023.csv', header='infer', sep=';', encoding='latin-1', low_memory=False)
df_municipios = pd.read_csv('municipios.csv', header='infer', sep=',', low_memory=False)

df_enem.to_parquet('enem_2023.parquet', index=False)
df_municipios.to_parquet('municipios.parquet', index=False)


del df_enem, df_municipios



In [2]:
import pandas as pd
import numpy as np
import time

start_leitura = time.time()
df_enem_parquet = pd.read_parquet('enem_2023.parquet')
tempo_leitura_parquet = time.time() - start_leitura

df_municipios_parquet = pd.read_parquet('municipios.parquet')

print(f"Dados prontos: {len(df_enem_parquet)} Participantes e {len(df_municipios_parquet)} Municipios")
print(f"Tempo de leitura do Parquet: {tempo_leitura_parquet:.4f} segundos")

# --- ESTRATÉGIA A: REDUCE-SIDE JOIN (Simulando Shuffle & Sort) ---
# O merge do pandas é eficiente, mas aqui ele simula o processo de alinhar
# duas tabelas grandes onde o sistema precisa "procurar" e "ordenar" as chaves.
start_shuffle = time.time()
resultado_shuffle = pd.merge(df_enem_parquet, df_municipios_parquet, on='CO_MUNICIPIO_PROVA')
tempo_shuffle_parquet = time.time() - start_shuffle

print(f"Tempo 'Reduce-side Join': {tempo_shuffle_parquet:.4f} segundos")

# --- ESTRATÉGIA B: MAP-SIDE JOIN (Simulando Broadcast & Distributed Cache) ---
# Nesta estratégia, a tabela pequena é enviada para o cache (dicionário)
# e a junção acontece instantaneamente durante o mapeamento da tabela grande.
start_map = time.time()

# 1. Simulação do Distributed Cache: enviamos a tabela menor para a RAM de todos os 'Mappers'
cache_municipios = df_municipios_parquet.set_index('CO_MUNICIPIO_PROVA')['capital'].to_dict()

# 2. Simulação do Map: o Join é feito via lookup direto (O(1)) sem mover a tabela fato pela rede
df_enem_parquet['capital'] = df_enem_parquet['CO_MUNICIPIO_PROVA'].map(cache_municipios)

tempo_map_parquet = time.time() - start_map
print(f"Tempo 'Map-side Join' (Vetorizado): {tempo_map_parquet:.4f} segundos")

print(f"\nResultado: O Map-side Join foi {tempo_shuffle_parquet/tempo_map_parquet:.1f}x mais rapido!")

Dados prontos: 3933955 Participantes e 5571 Municipios
Tempo de leitura do Parquet: 12.4373 segundos
Tempo 'Reduce-side Join': 1.2801 segundos
Tempo 'Map-side Join' (Vetorizado): 0.0604 segundos

Resultado: O Map-side Join foi 21.2x mais rapido!
